# Arlians training on Kaggle

Train the Arlians PPO policy on a **free Kaggle GPU**, with checkpointing so you can resume across sessions.

**Before you run:**
1. [Verify your phone](https://www.kaggle.com/settings) on Kaggle (required for GPU + internet).
2. Notebook settings (right sidebar): **Accelerator** → GPU T4 x2 (or P100), **Internet** → On.
3. Push this repo to GitHub (or upload a zip as a Kaggle dataset — see the config cell).
4. For long runs: **Save Version → Save & Run All (Commit)** so `/kaggle/working` outputs persist.

**Updating an existing Kaggle notebook:** you cannot merge two notebooks in one click. Either edit cells in place (copy the **Configuration** + **Train** + **Demo** cells from this file) or **File → Import Notebook** into a new notebook and re-attach your old output dataset (see Configuration → resume notes).

Full walkthrough: [docs/TRAINING_KAGGLE.md](https://github.com/adit-rah/arlians/blob/main/docs/TRAINING_KAGGLE.md)

## Configuration

Edit the values below, then run all cells top to bottom.

### Already committed a run?

1. Open that version’s **Output** → **Add as Data source** (you get `/kaggle/input/<name>/ckpt.pt`).
2. Set `RESUME_FROM_INPUT = True` and fix `PREVIOUS_CKPT_INPUT`.
3. If you changed **world size or agent counts**, use a **new** `CKPT_PATH` (e.g. `ckpt_512.pt`) and `RESUME_TRAINING = False` for a clean large-island run, *or* keep the old ckpt to warm-start weights (same policy architecture).
4. Set `SMOKE_TEST = False`, raise `TRAIN_UPDATES` (e.g. 10000), then **Save & Run All (Commit)** again.

In [ ]:
# --- GitHub clone (default) ---
GITHUB_USER = "adit-rah"
GITHUB_REPO = "arlians"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                 # only if repo is private

USE_DATASET_INPUT = False
DATASET_INPUT_PATH = "/kaggle/input/arlians"

WORLD_SEED = 42

# --- Resume from a previous committed notebook output ---
RESUME_FROM_INPUT = False         # True after Add as Data source on old Output
PREVIOUS_CKPT_INPUT = "/kaggle/input/<your-output-dataset>/ckpt.pt"

# --- Training ---
SMOKE_TEST = False                # True = short sanity run only
RESUME_TRAINING = True            # False = fresh random policy (new CKPT_PATH)

# Full run: large island, long job (resume across commits with higher TRAIN_UPDATES)
TRAIN_WIDTH = 512
TRAIN_AGENTS = 1024               # max entity slots (M)
TRAIN_INIT_AGENTS = 384           # starting population
TRAIN_UPDATES = 10000             # total PPO updates (raise across sessions)
TRAIN_HORIZON = 256               # sim steps per update
TRAIN_CHECKPOINT_EVERY = 50

# Smoke run (SMOKE_TEST=True): quick check before a long commit
SMOKE_WIDTH = 256
SMOKE_AGENTS = 256
SMOKE_INIT_AGENTS = 128
SMOKE_UPDATES = 30
SMOKE_HORIZON = 128

CKPT_PATH = "/kaggle/working/ckpt_512.pt"   # new path when changing world scale
METRICS_PATH = "/kaggle/working/metrics.jsonl"

# --- Survival-sim mode ---
# NO_RESPAWN=True turns off the population top-up crutch so lifespan reflects real
# survival skill. With it on, the cohort declines naturally during a rollout and a
# fresh cohort is re-seeded when the living population drops below init_agents//4
# (so training never stalls at zero agents). Watch the per-update `behavior` line:
# max_age/mean_age should climb, deaths should fall, as the policy learns to survive.
NO_RESPAWN = True

# --- Demo GIF (sparse pop on a large island) ---
DEMO_SIZE = 512                   # world side length (tiles)
DEMO_AGENTS = 96                  # initial living agents
DEMO_MAX_AGENTS = 192             # slot cap (must be >= DEMO_AGENTS)
DEMO_STEPS = 720                  # sim steps (~2 seasons at default period)
DEMO_OUT = "/kaggle/working/trained.gif"

## 1. Confirm GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none — set Accelerator to GPU T4 x2 in notebook settings.")

## 2. Get the code

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/arlians"

if USE_DATASET_INPUT:
    if not os.path.isdir(DATASET_INPUT_PATH):
        raise FileNotFoundError(
            f"Dataset not found: {DATASET_INPUT_PATH}. "
            "Upload the repo zip via + Add Input → Upload."
        )
    os.chdir(DATASET_INPUT_PATH)
    print("Using dataset input:", os.getcwd())
    print("Contents:", sorted(os.listdir("."))[:12], "...")
else:
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    if os.path.isdir(REPO_DIR):
        print("Repo already cloned; pulling latest...")
        subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"], stderr=subprocess.DEVNULL)
    else:
        subprocess.check_call([
            "git", "clone", "-b", GITHUB_BRANCH, clone_url, REPO_DIR
        ])
    os.chdir(REPO_DIR)
    print("Cloned to:", os.getcwd())
    subprocess.check_call(["git", "log", "--oneline", "-1"])

## 3. Install dependencies

PyTorch is preinstalled on Kaggle; we install numpy/scipy/opensimplex (imageio is usually present).

In [ ]:
!pip install -q "numpy>=1.26" "scipy>=1.12" "opensimplex>=0.4" imageio

## 4. (Optional) Restore checkpoint from a previous run

In [ ]:
import shutil

if RESUME_FROM_INPUT:
    if not os.path.isfile(PREVIOUS_CKPT_INPUT):
        raise FileNotFoundError(
            f"Checkpoint not found: {PREVIOUS_CKPT_INPUT}. "
            "Add your previous notebook output as a dataset (Output → Add as Data source)."
        )
    shutil.copy(PREVIOUS_CKPT_INPUT, CKPT_PATH)
    print(f"Copied checkpoint → {CKPT_PATH}")
else:
    print("Skipping input checkpoint copy (fresh run or resume from existing working ckpt).")

## 5. Generate the world

Deterministic for a given seed. Run once per seed (re-run only if you change `WORLD_SEED`).

In [ ]:
import subprocess

_gen_w = SMOKE_WIDTH if SMOKE_TEST else TRAIN_WIDTH
subprocess.check_call([
    "python", "generate.py",
    "--seed", str(WORLD_SEED),
    "--width", str(_gen_w),
    "--height", str(_gen_w),
])
print(f"exported world at seed={WORLD_SEED}, size={_gen_w}")

## 6. Train (PPO + checkpointing)

Watch the live log: `value_loss` falling, `mean_return` rising, `living` stable.

For long jobs use **Save Version → Save & Run All (Commit)**. Outputs land in `/kaggle/working/` (`ckpt.pt`, `metrics.jsonl`).

In [ ]:
import os
import subprocess

if SMOKE_TEST:
    train_args = [
        "--width", str(SMOKE_WIDTH),
        "--agents", str(SMOKE_AGENTS),
        "--init-agents", str(SMOKE_INIT_AGENTS),
        "--updates", str(SMOKE_UPDATES),
        "--horizon", str(SMOKE_HORIZON),
    ]
    ckpt_every = 10
    print("SMOKE_TEST=True — short sanity run.")
else:
    train_args = [
        "--width", str(TRAIN_WIDTH),
        "--agents", str(TRAIN_AGENTS),
        "--init-agents", str(TRAIN_INIT_AGENTS),
        "--updates", str(TRAIN_UPDATES),
        "--horizon", str(TRAIN_HORIZON),
    ]
    ckpt_every = TRAIN_CHECKPOINT_EVERY
    print(
        f"SMOKE_TEST=False — {TRAIN_WIDTH}x{TRAIN_WIDTH}, "
        f"{TRAIN_INIT_AGENTS}/{TRAIN_AGENTS} agents, "
        f"{TRAIN_UPDATES} updates x {TRAIN_HORIZON} steps/update"
    )

cmd = [
    "python", "train/run.py",
    *train_args,
    "--seed", str(WORLD_SEED),
    "--checkpoint", CKPT_PATH,
    "--checkpoint-every", str(ckpt_every),
    "--log", METRICS_PATH,
]
if RESUME_TRAINING:
    cmd.append("--resume")
if NO_RESPAWN:
    cmd.append("--no-respawn")

print(" ".join(cmd))
subprocess.check_call(cmd)

## 7. Visualize trained policy (optional)

Run after training finishes (or in a new session with `ckpt.pt` in `/kaggle/working` or attached as input). Download `trained.gif` from the **Output** panel.

In [ ]:
import os
import subprocess

if not os.path.isfile(CKPT_PATH):
    raise FileNotFoundError(
        f"No checkpoint at {CKPT_PATH}. Train first, or set RESUME_FROM_INPUT."
    )

subprocess.check_call([
    "python", "scripts/demo.py",
    "--checkpoint", CKPT_PATH,
    "--steps", str(DEMO_STEPS),
    "--agents", str(DEMO_AGENTS),
    "--max-agents", str(DEMO_MAX_AGENTS),
    "--size", str(DEMO_SIZE),
    "--seed", str(WORLD_SEED),
    "--out", DEMO_OUT,
])
print(f"GIF: {DEMO_OUT}  ({DEMO_AGENTS} agents, cap {DEMO_MAX_AGENTS}, {DEMO_SIZE}x{DEMO_SIZE})")

## Metrics quick look (optional)

In [ ]:
import json
import os

if not os.path.isfile(METRICS_PATH):
    print(f"No metrics at {METRICS_PATH} yet.")
else:
    rows = []
    with open(METRICS_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"{len(rows)} updates logged")
    if rows:
        last = rows[-1]
        keys = ["update", "mean_reward", "mean_return", "value_loss", "entropy", "n_living"]
        print({k: last.get(k) for k in keys if k in last})